In [3]:
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
import json
import re
import spacy
from spellchecker import SpellChecker

In [5]:
nlp = spacy.load("en_core_web_sm")
spell = SpellChecker()

In [6]:
abbreviations = {
    # --- People & Roles ---
    r"\bdoc\b": "doctor",
    r"\bdr\b": "doctor",
    r"\bpt\b": "patient",
    r"\bpts\b": "patients",
    r"\bnurse\b": "nurse",
    r"\bspecialist\b": "specialist",
    r"\bent\b": "ear nose and throat specialist",

    # --- Body Sides & Directions ---
    r"\blt\b": "left",
    r"\brt\b": "right",
    r"\bl\b": "left",
    r"\br\b": "right",
    r"\bbilat\b": "bilateral",
    r"\buni\b": "unilateral",
    r"\bleft ear\b": "left ear",
    r"\bright ear\b": "right ear",

    # --- Medical History & Context ---
    r"\bhx\b": "history",
    r"\bpmhx\b": "past medical history",
    r"\bfhx\b": "family history",
    r"\bsx\b": "symptoms",
    r"\bdx\b": "diagnosis",
    r"\btx\b": "treatment",
    r"\brx\b": "prescription",
    r"\bhtn\b": "hypertension",
    r"\bdm\b": "diabetes mellitus",
    r"\bobe\b": "obesity",
    r"\ballergies\b": "allergies",
    r"\bno known allergies\b": "no known allergies",
    r"\bnka\b": "no known allergies",

    # --- Symptoms & Conditions ---
    r"\bsob\b": "shortness of breath",
    r"\bha\b": "headache",
    r"\bcp\b": "chest pain",
    r"\bn&v\b": "nausea and vomiting",
    r"\bnv\b": "nausea and vomiting",
    r"\bh/a\b": "headache",
    r"\bearpain\b": "ear pain",
    r"\bearache\b": "ear pain",
    r"\btinnitus\b": "ringing in ears",
    r"\bvertigo\b": "dizziness and spinning",
    r"\bhoh\b": "hard of hearing",
    r"\bhl\b": "hearing loss",
    r"\betd\b": "eustachian tube dysfunction",
    r"\bom\b": "otitis media",
    r"\boe\b": "otitis externa",

    # --- Medications & Treatments ---
    r"\bmeds\b": "medications",
    r"\bmed\b": "medication",
    r"\botc\b": "over the counter",
    r"\bprn\b": "as needed",
    r"\bbid\b": "twice a day",
    r"\btid\b": "three times a day",
    r"\bqd\b": "once a day",
    r"\bqid\b": "four times a day",
    r"\bpo\b": "taken orally",
    r"\biv\b": "intravenous",
    r"\bab\b": "antibiotic",
    r"\babs\b": "antibiotics",
    r"\bpainkillers\b": "pain relief medications",
    r"\bpainkiller\b": "pain relief medication",
    r"\bibuprofen\b": "ibuprofen",
    r"\bparacetamol\b": "paracetamol",
    r"\beardrops\b": "ear drops",
    r"\bearwash\b": "ear irrigation",

    # --- Time & Duration ---
    r"\bsince am\b": "since morning",
    r"\bsince pm\b": "since evening",
    r"\bam\b": "morning",
    r"\bpm\b": "evening",
    r"\bd/c\b": "discharged",
    r"\bwks\b": "weeks",
    r"\bwk\b": "week",
    r"\bmos\b": "months",
    r"\bmo\b": "month",
    r"\byrs\b": "years",
    r"\byr\b": "year",
    r"\bdys\b": "days",
    r"\bdy\b": "day",
    r"\bhrs\b": "hours",
    r"\bhr\b": "hour",
    r"\bmins\b": "minutes",
    r"\bmin\b": "minute",
    r"\batm\b": "at the moment",
    r"\brn\b": "right now",
    r"\basap\b": "as soon as possible",

    # --- Casual / Chat Shorthand ---
    r"\bim\b": "i am",
    r"\bive\b": "i have",
    r"\bidк\b": "i do not know",
    r"\bidk\b": "i do not know",
    r"\bimo\b": "in my opinion",
    r"\bidk\b": "i do not know",
    r"\btbh\b": "to be honest",
    r"\bngl\b": "not going to lie",
    r"\bngl\b": "not going to lie",
    r"\bbtw\b": "by the way",
    r"\bfyi\b": "for your information",
    r"\bomg\b": "oh my god",
    r"\bsmh\b": "shaking my head",
    r"\bplz\b": "please",
    r"\bpls\b": "please",
    r"\bthx\b": "thanks",
    r"\bthnx\b": "thanks",
    r"\bty\b": "thank you",
    r"\btyvm\b": "thank you very much",
    r"\bok\b": "okay",
    r"\bokay\b": "okay",
    r"\bnp\b": "no problem",
    r"\bw/\b": "with",
    r"\bw/o\b": "without",
    r"\bb4\b": "before",
    r"\blmk\b": "let me know",
    r"\bsry\b": "sorry",
    r"\bsorry\b": "sorry",
    r"\bnbd\b": "no big deal",
    r"\bidc\b": "i do not care",
    r"\bwtf\b": "what the heck",
    r"\bomfg\b": "oh my god",
    r"\blol\b": "laughing",
    r"\blmao\b": "laughing a lot",
    r"\bsmth\b": "something",
    r"\bsth\b": "something",
    r"\babt\b": "about",
    r"\brly\b": "really",
    r"\brlly\b": "really",
    r"\bprob\b": "probably",
    r"\bprobs\b": "probably",
    r"\bprobably\b": "probably",
    r"\bdunno\b": "do not know",
    r"\bgonna\b": "going to",
    r"\bwanna\b": "want to",
    r"\bgotta\b": "got to",
    r"\bkinda\b": "kind of",
    r"\bsorta\b": "sort of",
    r"\bcuz\b": "because",
    r"\bcause\b": "because",
    r"\bcos\b": "because",
    r"\bbcz\b": "because",
    r"\bbc\b": "because",
    r"\bnvm\b": "never mind",
    r"\bnvmd\b": "never mind",
    r"\bafa\b": "as far as",
    r"\bafaik\b": "as far as i know",
    r"\bafaict\b": "as far as i can tell",
    r"\bggwp\b": "good going",
    r"\biykyk\b": "if you know you know",
    r"\bnote\b": "note",

    # --- Severity & Feeling ---
    r"\bvv\b": "very very",
    r"\bvery\b": "very",
    r"\bworst\b": "worst",
    r"\bbad\b": "bad",
    r"\bterrble\b": "terrible",
    r"\bunbearable\b": "unbearable",
    r"\bmild\b": "mild",
    r"\bmoderate\b": "moderate",
    r"\bsevere\b": "severe",
    r"\bchronic\b": "chronic",
    r"\bacute\b": "acute",
    r"\bintermittent\b": "intermittent",
    r"\boccasional\b": "occasional",
    r"\bconstant\b": "constant",
    r"\bpersistent\b": "persistent",
    r"\bon & off\b": "on and off",
    r"\bon n off\b": "on and off",
    r"\bon and off\b": "on and off",
}

In [7]:
symptomNormalizations = json.load(open("templates/symptomNormalizations.json", "r"))

In [8]:
relevantStopwords = json.load(open("templates/relevantStopwords.json", "r"))

In [10]:
for word in relevantStopwords:
    nlp.vocab[word].is_stop = False